# Example 6.1 (the Gaussian distributions) in CT+iDCI paper

## Setup libraries and helper functions

In [ ]:
# The libraries we will use
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde as GKDE
from scipy.stats import norm, beta, uniform, multivariate_normal
import scipy.io as sio
from scipy.stats import entropy
from tqdm import tqdm
import matplotlib.colors as mcolors

In [ ]:
from ipywidgets import interact, IntSlider
import ipywidgets as widgets

### Define exact KL divergence functions for Gaussians

In [ ]:
import scipy.linalg

def kl_multivariate_gaussians(mu1, cov1, mu2, cov2):
    """
    Computes the KL divergence KL(N1 || N2) between two multivariate Gaussians.
    N1 ~ N(mu1, cov1) -- "Truth" in this case
    N2 ~ N(mu2, cov2)
    """
    k = mu1.shape[0]
    
    # 1. Compute the log-determinants safely to prevent underflow/overflow
    # sign should be 1 for valid positive-definite covariance matrices
    sign1, logdet1 = np.linalg.slogdet(cov1)
    sign2, logdet2 = np.linalg.slogdet(cov2)
    
    # 2. Solve the linear systems using Cholesky factor of cov2 for stability
    # This replaces the need for an explicit and unstable matrix inversion (cov2^-1)
    cholesky_factor = scipy.linalg.cho_factor(cov2)
    
    # Compute Tr(cov2^-1 * cov1)
    inv_cov2_cov1 = scipy.linalg.cho_solve(cholesky_factor, cov1)
    trace_term = np.trace(inv_cov2_cov1)
    
    # Compute (mu2 - mu1)^T * cov2^-1 * (mu2 - mu1)
    diff = mu2 - mu1
    mahalanobis_term = diff.T @ scipy.linalg.cho_solve(cholesky_factor, diff)
    
    # 3. Combine terms according to the analytical formula
    kl_div = 0.5 * (trace_term + mahalanobis_term - k + logdet2 - logdet1)
    
    return kl_div

In [ ]:
def kl_divergence_gaussians(mu1, sigma1, mu2, sigma2):
    """
    Calculate the KL divergence between two univariate Gaussians: P || Q
    P ~ N(mu1, sigma1^2)
    Q ~ N(mu2, sigma2^2)
    """
    # Ensure standard deviations are positive
    if sigma1 <= 0 or sigma2 <= 0:
        raise ValueError("Standard deviations must be greater than 0.")
        
    term1 = np.log(sigma2 / sigma1)
    term2 = (sigma1**2 + (mu1 - mu2)**2) / (2 * sigma2**2)
    term3 = -0.5
    
    return term1 + term2 + term3

### Define functions used for fitting Gaussians

In [ ]:
def weighted_Gaussian_params(x, w):
    weighted_mean = np.average(x, 
                               weights = w,
                               axis=0)
    weighted_cov = np.cov(x, 
                          aweights = w,
                          rowvar = False,
                          ddof = 1)
    return weighted_mean, weighted_cov

In [ ]:
def weighted_correlation(x, y, w):
    """Calculate the weighted Pearson correlation coefficient between two vectors."""
    x, y, w = np.array(x), np.array(y), np.array(w)
    
    # 1. Calculate weighted means
    mean_x = np.sum(x * w) / np.sum(w)
    mean_y = np.sum(y * w) / np.sum(w)
    
    # 2. Calculate weighted variances and covariance
    cov_xy = np.sum(w * (x - mean_x) * (y - mean_y)) / np.sum(w)
    cov_xx = np.sum(w * (x - mean_x) ** 2) / np.sum(w)
    cov_yy = np.sum(w * (y - mean_y) ** 2) / np.sum(w)
    
    # 3. Calculate weighted correlation
    return cov_xy / np.sqrt(cov_xx * cov_yy)

### Define iDCI function 

Note that this iDCI function differs from those in other notebooks in that it uses parametric estimates of the predicted densities and the exact KL divergence functions to check tolerances.

In [ ]:
def iterative_DCI(Q_pred_vals, observed_densities, obs_means, obs_covs,
                  num_epochs=1, r_init=1, 
                  pred_tol=1e-12, ratio_tol=0.1, 
                  KL_tol=1e-7, KL_update_rel_tol = 1e-2,
                  verbose=False, QoI_spaces=[]):
    '''
    Assume Q_pred_vals is a (num_samples x QoI_num) array of QoI vals 
    associated with initial parameter samples and observed_densities 
    is a list of observed QoI densities for each QoI map (coming from a
    data generating distribution)
    '''
    num_samples, num_QoI = np.shape(Q_pred_vals)

    if len(QoI_spaces) == 0:
        QoI_spaces = []
        for i in range(num_QoI):
            QoI_spaces.append([i])
    
    true_obs_probs = []
    for d in range(len(QoI_spaces)):
        true_obs_probs.append(observed_densities[d](Q_pred_vals[:,QoI_spaces[d]]))
    
    with tqdm(total=100) as pbar:    
        
        if np.size(r_init)==1:  # No initial weighting (possibly from prior iterations)
            rs = np.ones((num_samples, len(QoI_spaces), num_epochs))
            r_current = np.ones(num_samples)
        else:
            r_current = np.copy(r_init)
            rs = np.ones((num_samples, len(QoI_spaces), num_epochs))
            
        kl_from_observed_marginal = np.zeros((len(QoI_spaces), num_epochs))
        
        for k in range(num_epochs):
            
            if verbose is True:
                print('Starting epoch #', k)
            
            for d in range(len(QoI_spaces)):
                r_current /= np.mean(r_current)
                if verbose is True: 
                    print('Iteration #', d)
                    print('Some current r-values:', r_current[0:3])

                pred_mean, pred_cov = weighted_Gaussian_params(Q_pred_vals[:, QoI_spaces[d]],
                                                               r_current)
                
                pred_dens = lambda x : multivariate_normal.pdf(x, 
                                                               mean = pred_mean,
                                                               cov = pred_cov)
                
                rs_update = np.divide(observed_densities[d](Q_pred_vals[:,QoI_spaces[d]]),
                                      pred_dens(Q_pred_vals[:,QoI_spaces[d]]))
                
                r_current *= rs_update
                
                rs[:,d,k] = r_current
                
                if np.abs(np.mean(r_current)-1.0)>ratio_tol:
                    print('Outside of ratio tolerance at epoch #', k+1, ' iteration #', d+1)
                    return rs[:,:d+1,:k+1], k+1, d+1, kl_from_observed_marginal[:,k]
                pbar.update(100/(len(QoI_spaces)*num_epochs))
            
            for d in range(len(QoI_spaces)):
                pred_mean, pred_cov = weighted_Gaussian_params(Q_pred_vals[:,QoI_spaces[d]], 
                                                               r_current)

                if len(QoI_spaces[d]) == 1:
                    kl_from_observed_marginal[d, k] = kl_divergence_gaussians(obs_means[d],
                                                                              obs_covs[d],
                                                                              pred_mean,
                                                                              pred_cov)
                else:
                    kl_from_observed_marginal[d, k] = kl_multivariate_gaussians(obs_means[d],
                                                                            obs_covs[d],
                                                                            pred_mean, 
                                                                            pred_cov)
            
            if verbose is True:
                print('Epoch #', k)
                print('KL divergences from observed marginals', kl_from_observed_marginal[:,k])
            
            if all( kl < KL_tol for kl in kl_from_observed_marginal[:,k]):
                print('KL divergences from observed marginals all within tolerance at epoch #', k+1)
                return rs[:,:,:k+1], k+1, d+1, kl_from_observed_marginal
            
            if k>0:
                kl_relative_update = np.abs(kl_from_observed_marginal[:,k-1] - kl_from_observed_marginal[:,k]) /\
                                    kl_from_observed_marginal[:,k-1]
                if all(kl_rel < KL_update_rel_tol for kl_rel in kl_relative_update):
                    print('KL divergences from observed marginals are not sufficiently updated at epoch #', k+1)
                    return rs[:,:,:k+1], k+1, d+1, kl_from_observed_marginal
                    
    return rs, k+1, d+1, kl_from_observed_marginal

### Copula transformation function

Note that this copula transformation differs from others in that to map to the copula coordinate system we use the Gaussian CDF within the `norm` class as opposed to an integrate function from KDE estimates of non-parametric observed distributions.

In [ ]:
def copula_transformation(rs, Q_pred_vals):
    
    rho_hat_pred = weighted_correlation(Q_pred_vals[:,0], Q_pred_vals[:,1], rs)
    rho_hat_obs = np.corrcoef(Q_obs_vals[:,0], Q_obs_vals[:,1])[0,1]

    # Use CDFs of the observed distributions to map to the shared
    # copula coordinate system
    Fx_interp = lambda x: norm.cdf(x, loc=obs_means[0], scale=np.sqrt(obs_covs[0]))
    Fy_interp = lambda y: norm.cdf(y, loc=obs_means[1], scale=np.sqrt(obs_covs[1]))

    # Define Gaussian copula function
    def copula_Gaussian(rho, data_1, data_2):
        return multivariate_normal.pdf(np.dstack((data_1, data_2)),
                                       cov = [[1, rho], [rho, 1]]                                        )
    
    # Transform predictions to pseudo-obs space 
    # this requires the weighted eCDFs in general
    u = Fx_interp(Q_pred_vals[:,0])
    v = Fy_interp(Q_pred_vals[:,1])
    
    # Now ensure we avoid the boundary of the unit hyperbox
    eps = 1e-15
    u = np.clip(u, eps, 1-eps)
    v = np.clip(v, eps, 1-eps)
    
    # Transform to Gaussian scores
    z = norm.ppf(u)
    q = norm.ppf(v)
    
    # Evaluate the predicted and observed copulas on their shared coordinate system
    copula_obs = copula_Gaussian(rho_hat_obs, z, q)
    copula_pred = copula_Gaussian(rho_hat_pred, z, q)
    
    copula_r = np.divide(copula_obs, copula_pred)

    copula_transformed_r = copula_r * rs

    return copula_transformed_r, copula_r

### Create 2d plotting functions

`plot_epoch_2d_params` plots the updates and QoI densities as a function of epoch number

`plot_2d_params_compare` allows us to compare different parameter distributions

Note that these functions differ from those that appear in other notebooks in that parametric density estimates are computed/plotted.

In [ ]:
def plot_epoch_2d_params(fignum, epoch_num, observed_densities, rs, 
                         fix_cbar_ranges=False, cmap='hot_r', savefig=False,
                         filename='file.png', show_QoI=True, num_points=100):

    x = np.linspace(-3, 3, num_points)
    y = np.linspace(-3, 3, num_points)    
    X, Y = np.meshgrid(x, y)
    plotting_params = np.vstack([X.ravel(), Y.ravel()]).T

    # Construct updated densities for particular iterations of epoch
    up_mean, up_cov = weighted_Gaussian_params(params_init, rs[:,-1,epoch_num-1]) 
    
    update_dens = lambda x : multivariate_normal.pdf(x, 
                                            mean = up_mean,
                                            cov = up_cov)
    
    dg_mean = np.average(params_dg, axis=0)
    dg_cov = np.cov(params_dg, rowvar=False, ddof=1)
    dg_dens = lambda x : multivariate_normal.pdf(x,
                                                 mean=dg_mean,
                                                 cov=dg_cov)

    # Plot settings defined as dict
    base_cmap = plt.get_cmap(cmap)
    shared_levels=20
    color_matrix = base_cmap(np.linspace(0, 1, shared_levels-1))
    
    # Force the first 4 rows (bottom 4 levels) to be white
    color_matrix[0, :] = [1.0, 1.0, 1.0, 1.0] # [R, G, B, Alpha]
    custom_cmap = mcolors.ListedColormap(color_matrix)
    # Plot settings defined as dict
    plot_dict = {'cmap' : custom_cmap}
    
    if fix_cbar_ranges is True:
        true_vals = dg_dens(plotting_params)
        true_max = np.max(true_vals)
        true_min = np.min(true_vals)
        plot_dict['vmin'] = true_min
        plot_dict['vmax'] = true_max
        shared_levels = np.linspace(true_min, true_max, shared_levels)

    # Create plots
    if show_QoI:
        fig = plt.figure(num=fignum, figsize=(9, 6), tight_layout=True)
        fig.clf()
        gs = fig.add_gridspec(6,2)
        ax1 = fig.add_subplot(gs[0:4,0])
        ax4 = fig.add_subplot(gs[0:4,1])
        ax5 = fig.add_subplot(gs[4:,0])
        ax6 = fig.add_subplot(gs[4:,1])
    else:
        fig = plt.figure(num=fignum, figsize=(9, 4), tight_layout=True)
        fig.clf()
        gs = fig.add_gridspec(6,2)
        ax1 = fig.add_subplot(gs[0:,0])
        ax4 = fig.add_subplot(gs[0:,1])
        

    ax1.set_aspect('equal', adjustable='box')
    cf = ax1.contourf(X, Y, update_dens(plotting_params).reshape(len(y), len(x)),
                      levels=shared_levels,
                      **plot_dict)
    title_str = 'iDCI after epoch ' + str(epoch_num)
    ax1.set_title(title_str)
    ax1.set_xlabel(r'$\lambda_1$')
    ax1.set_ylabel(r'$\lambda_2$')
    plt.colorbar(cf, ax=ax1, fraction=0.046, pad=0.04)

    
    cf4 = ax4.contourf(X, Y, dg_dens(plotting_params).reshape(len(y), len(x)),
                      levels=shared_levels,
                      **plot_dict)
    ax4.set_aspect('equal', adjustable='box')
    ax4.set_title('Sample DG Distribution')
    ax4.set_xlabel(r'$\lambda_1$')
    ax4.set_ylabel(r'$\lambda_1$')
    plt.colorbar(cf4, ax=ax4, fraction=0.046, pad=0.04)

    if show_QoI:
        weighted_pred_mean = np.average(Q_pred_vals[:,0], 
                                             weights = rs[:,-1,epoch_num-1],
                                             axis=0)
        weighted_pred_cov = np.cov(Q_pred_vals[:,0].T, 
                                        aweights = rs[:,-1,epoch_num-1],
                                        rowvar = False,
                                        ddof = 1)
        pred_dens = lambda x : multivariate_normal.pdf(x, 
                                                       mean = weighted_pred_mean,
                                                       cov = weighted_pred_cov)
        min_Q = np.min(Q_pred_vals[:,0])
        max_Q = np.max(Q_pred_vals[:,0])
        plt_Qs = np.linspace(min_Q-0.05*(max_Q-min_Q), max_Q+0.05*(max_Q-min_Q), 101)
        ax5.plot(plt_Qs, pred_dens(plt_Qs), linestyle=':', linewidth=2, label='PF of Update')
        ax5.plot(plt_Qs, observed_densities[0](plt_Qs), label='Observed')
        ax5.set_xlabel(r'$Q_1$')
        ax5.set_title('$Q_1$ densities after epoch ' + str(epoch_num))
        ax5.legend()
        
        weighted_pred_mean = np.average(Q_pred_vals[:,1], 
                                             weights = rs[:,-1,epoch_num-1],
                                             axis=0)
        weighted_pred_cov = np.cov(Q_pred_vals[:,1].T, 
                                        aweights = rs[:,-1,epoch_num-1],
                                        rowvar = False,
                                        ddof = 1)
        pred_dens = lambda x : multivariate_normal.pdf(x, 
                                                       mean = weighted_pred_mean,
                                                       cov = weighted_pred_cov)
        min_Q = np.min(Q_pred_vals[:,1])
        max_Q = np.max(Q_pred_vals[:,1])
        plt_Qs = np.linspace(min_Q-0.05*(max_Q-min_Q), max_Q+0.05*(max_Q-min_Q), 101)
        ax6.plot(plt_Qs, pred_dens(plt_Qs), linestyle=':', linewidth=2, label='PF of Update')
        ax6.plot(plt_Qs, observed_densities[1](plt_Qs), label='Observed')
        ax6.set_xlabel(r'$Q_2$')
        ax6.set_title('$Q_2$ densities after epoch ' + str(epoch_num))
        ax6.legend()

    plt.tight_layout(pad=0.25)
    if savefig is True:
        plt.savefig(filename, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

In [ ]:
def plot_2d_params_compare(fignum, rs1, rs2, which_params=[0, 1], 
                           num_points=100, cmap='viridis_r',
                           ax1_title = 'iDCI Solution', 
                           ax2_title = 'CT+iDCI Solution',
                           fix_cbar_ranges = True,
                           savefig=False,
                           filename='file.png'):

    x = np.linspace(-3, 3, num_points)
    y = np.linspace(-3, 3, num_points)    
    X, Y = np.meshgrid(x, y)
    plotting_params = np.vstack([X.ravel(), Y.ravel()]).T

    Q_vals_1 = Q_maps[0](plotting_params)
    Q_vals_2 = Q_maps[1](plotting_params)

    # Construct updated densities for final rs1 and rs2 values
    up_mean_1, up_cov_1 = weighted_Gaussian_params(params_init, rs1) 
    
    update_dens_1 = lambda x : multivariate_normal.pdf(x, 
                                            mean = up_mean_1,
                                            cov = up_cov_1)

    # Construct updated densities for particular iterations of epoch
    up_mean_2, up_cov_2 = weighted_Gaussian_params(params_init, rs2) 
    
    update_dens_2 = lambda x : multivariate_normal.pdf(x, 
                                            mean = up_mean_2,
                                            cov = up_cov_2)
    
    dg_mean = np.average(params_dg, axis=0)
    dg_cov = np.cov(params_dg, rowvar=False, ddof=1)
    dg_dens = lambda x : multivariate_normal.pdf(x,
                                                 mean=dg_mean,
                                                 cov=dg_cov)

    # Plot settings defined as dict
    base_cmap = plt.get_cmap(cmap)
    shared_levels=20
    color_matrix = base_cmap(np.linspace(0, 1, shared_levels-1))
    
    # Force the first 4 rows (bottom 4 levels) to be white
    color_matrix[0, :] = [1.0, 1.0, 1.0, 1.0] # [R, G, B, Alpha]
    custom_cmap = mcolors.ListedColormap(color_matrix)
    # Plot settings defined as dict
    plot_dict = {'cmap' : custom_cmap}
    
    if fix_cbar_ranges is True:
        true_vals = dg_dens(plotting_params)
        true_max = np.max(true_vals)
        true_min = np.min(true_vals)
        plot_dict['vmin'] = true_min
        plot_dict['vmax'] = true_max
        shared_levels = np.linspace(true_min, true_max, shared_levels)

    init_contours = lambda x, y: np.sqrt(x**2 + y**2)
    
    # Create plots
    fig, axs = plt.subplots(1, 3, num=fignum, figsize=(9, 3), 
                            sharey='row', constrained_layout=True)

    ax1 = axs[0]
    ax1.set_aspect('equal', adjustable='box')
    ax1.contour(X, Y, init_contours(X,Y), levels=[1, 2, 3], 
                colors='black', alpha=0.1, linewidths=1)
    ax1.contour(X, Y, Q_vals_1.reshape(len(y), len(x)), levels=5,
                colors='red', alpha=0.5, linewidths=1,
               linestyles='dashed')
    ax1.contour(X, Y, Q_vals_2.reshape(len(y), len(x)), levels=5,
                colors='blue', alpha=0.5, linewidths=1,
               linestyles='dotted')
    cf = ax1.contourf(X, Y, update_dens_1(plotting_params).reshape(len(y), len(x)),
                      levels=shared_levels, alpha=1,
                      **plot_dict)
    title_str = ax1_title
    ax1.set_title(title_str)
    ax1.set_xlabel(r'$\lambda_1$')
    ax1.set_ylabel(r'$\lambda_2$')
    
    ax2 = axs[1]
    ax2.set_aspect('equal', adjustable='box')
    ax2.contour(X, Y, init_contours(X,Y), levels=[1, 2, 3] , colors='black', alpha=0.1, linewidths=1)
    ax2.contour(X, Y, Q_vals_1.reshape(len(y), len(x)), levels=5,
                colors='red', alpha=0.5, linewidths=1,
               linestyles='dashed')
    ax2.contour(X, Y, Q_vals_2.reshape(len(y), len(x)), levels=5,
                colors='blue', alpha=0.5, linewidths=1,
               linestyles='dotted')
    cf = ax2.contourf(X, Y, update_dens_2(plotting_params).reshape(len(y), len(x)),
                      levels=shared_levels,
                      **plot_dict)
    title_str = ax2_title
    ax2.set_title(title_str)
    ax2.set_xlabel(r'$\lambda_1$')
    ax2.set_ylabel(r'$\lambda_2$')

    ax3 = axs[2]
    ax3.set_aspect('equal', adjustable='box')
    ax3.contour(X, Y, init_contours(X,Y), levels=[1, 2, 3] , colors='black', alpha=0.1, linewidths=1)
    cf = ax3.contourf(X, Y, dg_dens(plotting_params).reshape(len(y), len(x)),
                      levels=shared_levels,
                      **plot_dict)
    title_str = 'Sample DG Distribution'
    ax3.set_title(title_str)
    ax3.set_xlabel(r'$\lambda_1$')
    ax3.set_ylabel(r'$\lambda_2$')
    
    fig.colorbar(cf, ax=[ax1, ax2, ax3], shrink=0.8)
    
    if savefig is True:
        plt.savefig(filename, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

## Setup parameter samples

In [ ]:
# Initialize random seed for reproducibility
np.random.seed(121212)

num_init_samples = int(1e4)
param_dim = 2

params_init = np.random.multivariate_normal(mean=(0, 0), cov=np.array([[1, 0], [0, 1]]), size=(num_init_samples))

In [ ]:
num_dg_samples = int(1e3)

params_dg = np.random.multivariate_normal(mean=(.5,.5), cov=np.array([[0.5, .4],[.4, 0.5]]), size=(num_dg_samples))

In [ ]:
plt.figure(num=1, figsize=(3,3))
plt.clf()
plt.scatter(params_init[:,0], params_init[:,1], label='Initial')
plt.scatter(params_dg[:,0], params_dg[:,1], marker='+',label='DG')
plt.xlabel(r'$\lambda_1$')
plt.ylabel(r'$\lambda_2$')
plt.legend()
plt.tight_layout(pad=.25)
# plt.savefig('ex1-2d-Gaussian-params.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

### Choose a skewed or non-skewed map

In [ ]:
#2D Linear maps

skewed = True

if skewed:
    Q_maps = [lambda x: 2*x[:,0] + x[:,1], lambda x: 2.5*x[:,0]+0.5*x[:,1]]  # Copulas have significant impact
else:
    Q_maps = [lambda x: 2*x[:,0] + x[:,1], lambda x: x[:,0]-x[:,1]]  # Copulas have minimal impact

In [ ]:
num_QoI = len(Q_maps)
Q_pred_vals = np.zeros((num_init_samples, num_QoI))
Q_obs_vals = np.zeros((num_dg_samples, num_QoI))

for d in range(num_QoI):
    Q_pred_vals[:, d] = Q_maps[d](params_init)
    Q_obs_vals[:, d] = Q_maps[d](params_dg)

In [ ]:
fig = plt.figure(num=2, figsize=(3,3))
fig.clf()
plt.scatter(Q_pred_vals[:,0], Q_pred_vals[:,1], label='Predicted')
plt.scatter(Q_obs_vals[:,0], Q_obs_vals[:,1], marker='+', label='Observed')
plt.xlabel(r'$Q_1$')
plt.ylabel(r'$Q_2$')
plt.legend()
plt.tight_layout(pad=.25)
# plt.savefig('ex1-2d-Gaussian-non-skewed-QoI.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

## Construct iDCI solution

In [ ]:
observed_densities = []
obs_means = []
obs_covs = []
for d in range(num_QoI):
    obs_means.append(np.average(Q_obs_vals[:,d], axis=0))
    obs_covs.append(np.cov(Q_obs_vals[:,d].T, ddof = 1))
    observed_densities.append(lambda x, mu=obs_means[d], cov=obs_covs[d] : multivariate_normal.pdf(x, 
                                                  mean = mu,
                                                  cov = cov))

In [ ]:
num_epochs = 100

rs_iDCI, last_epoch, last_iter, kl_epochs = iterative_DCI(Q_pred_vals, observed_densities, 
                                                     obs_means, obs_covs,
                                                     num_epochs = num_epochs, r_init = 1, 
                                                     ratio_tol=0.1, KL_tol=1e-5)

In [ ]:
print(np.mean(rs_iDCI[:,0,:], axis=0))  # looking at diagnostic at end of each epoch

In [ ]:
fignum = 4

interact(plot_epoch_2d_params, 
     fignum = widgets.fixed(fignum),
     epoch_num=IntSlider(min=1, max=last_epoch, steps=1),
     observed_densities=widgets.fixed(observed_densities),
     rs = widgets.fixed(rs_iDCI),
     fix_cbar_ranges=widgets.Checkbox(value=True, 
                                      description='KDE of DG defines all colorbars',
                                      width='auto',
                                      disabled=False),
     cmap=widgets.Text('hot_r'),
     savefig=widgets.Dropdown(options=[True, False], value=False),
     filename=widgets.Text('file.png'))

# Copula transformation

In [ ]:
rs_CT_iDCI, _ = copula_transformation(rs_iDCI[:,-1,-1], Q_pred_vals)

In [ ]:
np.mean(rs_CT_iDCI)  # Check diagnostic

## Compare iDCI to CT+iDCI

In [ ]:
plt.figure(num = 6)
plt.clf()

plt.scatter(rs_iDCI[:,-1,-1], rs_CT_iDCI)
plt.xlabel('iDCI $r$-values')
plt.ylabel('CT-iDCI $r$-values')
# plt.axis('square')
# plt.savefig('ex1-2d-Gaussian-nonskewed-QoI-CT-iDCI-r-values.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
fignum=8

if skewed:
    fname = 'ex1-Gaussian-skewed-QoI-param-densities'
else:
    fname = 'ex1-Gaussian-nonskewed-QoI-param-densities'

plot_2d_params_compare(fignum, rs_iDCI[:,-1,-1], rs_CT_iDCI, which_params=[0, 1], cmap='viridis_r',
                       savefig=True, filename=fname)

## Evaluate KL divergences

In [ ]:
mu_dg = np.mean(params_dg, axis=0)
cov_dg = np.cov(params_dg.T)

In [ ]:
mu_up_iDCI, cov_up_iDCI = weighted_Gaussian_params(params_init, rs_iDCI[:,-1,-1])

In [ ]:
KL_iDCI = kl_multivariate_gaussians(mu_dg, cov_dg, mu_up_iDCI, cov_up_iDCI)
print(KL_iDCI)

In [ ]:
mu_up_CT_iDCI, cov_up_CT_iDCI = weighted_Gaussian_params(params_init, rs_CT_iDCI)

In [ ]:
KL_CT_iDCI = kl_multivariate_gaussians(mu_dg, cov_dg, mu_up_CT_iDCI, cov_up_CT_iDCI)
print(KL_CT_iDCI)

In [ ]:
print(KL_iDCI / KL_CT_iDCI)